# 因子收益估计与预测

联合估计八个风格因子和八个选定 Alpha 的日收益系数，再生成滚动预测。输入为行情状态、收益标签和两类暴露，输出为 `factor_return.parquet`。


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

## 文件路径与参数

自动向上定位项目根目录，使用项目相对路径读写文件；内核工作目录应位于本项目内。下方同时定义本步骤使用的参数。


In [ ]:
PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'environment.yml').is_file() and (path / 'src').is_dir()
)

FACTOR_RETURN_START = pd.Timestamp("20230101")

PANEL = PROJECT_ROOT / "data/processed/panel.parquet"
RETURN_T2 = PROJECT_ROOT / "src/return_t2/return_t2.parquet"
STYLE_FACTOR_EXPOSURE = PROJECT_ROOT / "src/factor_exposure/style_factor/style_factor_exposure.parquet"
ALPHA_FACTOR_EXPOSURE = PROJECT_ROOT / "src/factor_exposure/alpha_factor/alpha_factor_exposure.parquet"
FACTOR_RETURN_OUTPUT = PROJECT_ROOT / "src/factor_return/factor_return.parquet"


## 读取并整理模型输入

读取股票状态、风格暴露、中性化 Alpha 和 `return_t2`。收益回归使用事后标签，交易预测仅使用后续计算的滞后预测列。


In [ ]:
panel = pd.read_parquet(PANEL)
alpha_factor_exposure = pd.read_parquet(ALPHA_FACTOR_EXPOSURE)
return_t2 = pd.read_parquet(RETURN_T2)
style_factor_exposure = pd.read_parquet(STYLE_FACTOR_EXPOSURE)

In [ ]:
panel = panel[['trade_date', 'ts_code', 'ST', 'suspend']]

## 选定 Alpha 与统一方向

选取八个 Alpha，对 `alpha80`、`alpha112`、`alpha177`、`alpha110` 反向；股票预测阶段使用相同处理，保证系数与暴露方向一致。


In [ ]:
alpha_factor_exposure = alpha_factor_exposure[['trade_date', 'ts_code', 
                                 'alpha42', 'alpha80', 
                                 'alpha112', 'alpha177', 
                                 'alpha1', 'alpha110', 
                                 'alpha132', 'alpha12']].copy()

alpha_factor_exposure[['alpha80', 'alpha112', 'alpha177', 'alpha110']] = -alpha_factor_exposure[['alpha80', 'alpha112', 'alpha177', 'alpha110']]

## 对齐每日回归数据

按日期和股票合并各输入，仅保留因子收益估计起始日及之后的记录。


In [ ]:
factor_exposure = pd.merge(panel, style_factor_exposure, on = ['trade_date', 'ts_code'])
factor_exposure = pd.merge(factor_exposure, alpha_factor_exposure, on = ['trade_date', 'ts_code'])
factor_exposure = pd.merge(factor_exposure, return_t2, on = ['trade_date', 'ts_code'])

factor_exposure = factor_exposure[(factor_exposure['trade_date'] >= FACTOR_RETURN_START)].copy().reset_index(drop=True)

In [ ]:
factor_exposure.head()

## 逐日联合回归

八个风格暴露与八个 Alpha 共同作为解释变量，不带截距：

$$
\mathrm{return\_t2}_{i,t}=\boldsymbol x_{i,t}^{\top}\boldsymbol f_t+\varepsilon_{i,t}.
$$

每日只使用非 ST、非停牌且全部暴露与标签有效的股票，样本数须大于 16。系数按信号日记录，其对应收益在后日开盘实现。


In [ ]:
alpha_cols = ['alpha42', 'alpha80', 'alpha112', 'alpha177',
              'alpha1', 'alpha110', 'alpha132', 'alpha12']
style_cols = ['size', 'mom', 'vol1', 'vol2', 'liq1', 'liq2', 'beta', 'value']
exposure_cols = style_cols + alpha_cols
regression_cols = exposure_cols + ['return_t2']

factor_return = []
model = LinearRegression(fit_intercept=False)

In [ ]:
for date, group in factor_exposure.groupby('trade_date'):
    group = group.loc[group['ST'].eq(False) & group['suspend'].eq(False)]
    group = group.loc[np.isfinite(group[regression_cols]).all(axis=1)]
    if len(group) <= len(exposure_cols):
        continue

    X = group[exposure_cols]
    y = group['return_t2']
    model.fit(X, y)

    coefficients = pd.Series(model.coef_, index=exposure_cols)
    factor_return.append({
        'trade_date': date,
        **coefficients.to_dict()
    })

## 整理因子收益表

将每日回归系数整理为一行一个信号日、每列一个因子的表，并按日期升序排列。


In [ ]:
factor_return = pd.DataFrame(factor_return, columns=['trade_date'] + exposure_cols)
factor_return = factor_return.sort_values('trade_date').reset_index(drop=True)

In [ ]:
factor_return.head()

## 全部因子收益预测

八个风格因子和八个 Alpha 均使用相同方法，按日期升序先 `.shift(2)`，再 `.rolling(250).mean()`：

$$
\widehat f_{j,t}=\frac{1}{250}\sum_{k=0}^{249}f_{j,t-2-k}
$$

`factor_return_pred` 保留日期和全部 16 个预测因子列，不修改原始 `factor_return`。使用完整窗口、不填充缺失，最早第 252 条记录产生预测。位移和窗口按收益表行数计算，在日期连续时对应市场交易日。

In [ ]:
factor_return_pred = (
    factor_return.sort_values('trade_date').reset_index(drop=True).copy()
)
factor_return_pred[exposure_cols] = (
    factor_return_pred[exposure_cols].shift(2).rolling(250).mean()
)

In [ ]:
factor_return_pred.tail()

## 合并保存实际因子收益与预测值

`factor_return_combined` 按 `trade_date` 合并全部 16 个因子的实际收益和预测收益。实际列名不变，预测列统一加 `_pre`，每个因子的两列相邻排列，例如 `size、size_pre` 和 `alpha42、alpha42_pre`。最终包含日期及 32 个因子数值列，只保存这一份合并数据至 `FACTOR_RETURN_OUTPUT` 指定的 `factor_return.parquet`。

In [ ]:
factor_return_combined = factor_return.merge(
    factor_return_pred.rename(
        columns={col: f'{col}_pre' for col in exposure_cols}
    ),
    on='trade_date', how='left', validate='one_to_one'
)

combined_cols = ['trade_date'] + [
    name for col in exposure_cols for name in (col, f'{col}_pre')
]
factor_return_combined = (
    factor_return_combined[combined_cols]
    .sort_values('trade_date').reset_index(drop=True)
)

In [ ]:
factor_return_combined.tail()

## 保存合并数据

写入 `FACTOR_RETURN_OUTPUT`，共日期及 16 组实际值、预测值。股票预测读取 `_pre` 列；实际系数不能直接作为同日可知的预测。


In [ ]:
factor_return_combined.to_parquet(FACTOR_RETURN_OUTPUT, index=False)

## 报告图表：因子收益与预测

风格和 Alpha 分为两张图，浅灰细线展示原始每日因子收益，蓝线展示模型预测，不对实际收益进行移动平均。横轴为信号日，实际系数对应收益在后日开盘实现。图片保存到 `src/figures/`。


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

FIGURE_DIR = PROJECT_ROOT / 'src/figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10})

# Separate style and Alpha panels keep the report figures readable.
report_factor_data = factor_return_combined.set_index('trade_date').sort_index()
for report_group, report_cols in [('style', style_cols), ('alpha', alpha_cols)]:
    fig, axes = plt.subplots(4, 2, figsize=(13, 11), sharex=True, layout='constrained')
    for ax, col in zip(axes.flat, report_cols):
        ax.plot(report_factor_data.index,
                report_factor_data[col],
                color='#cbd5e1', linewidth=.5, label='Realized daily factor return')
        ax.plot(report_factor_data.index, report_factor_data[f'{col}_pre'],
                color='#2563eb', linewidth=1.3, label='Forecast: lag 2, mean 250')
        ax.axhline(0, color='#475569', linewidth=.6)
        ax.set_title(col, loc='left')
        ax.yaxis.set_major_formatter(PercentFormatter(1))
        ax.tick_params(axis='x', labelrotation=30)
        ax.grid(alpha=.2)
    for ax in list(axes.flat)[len(report_cols):]:
        ax.set_visible(False)
    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='outside lower center', ncol=2, frameon=False)
    fig.suptitle(f'{report_group.title()} factor returns and forecasts; x-axis = signal date')
    fig.savefig(FIGURE_DIR / f'factor_returns_{report_group}.png', dpi=200)
    plt.show()
